Here we will create  YouTube chatbot where we can chat on any YouTube video content. 
- This chatbot will take transcript of that video as input and we can then do question answer with it.
- Either we can crate a chrome plugin, so that directly a chatbot will open in YouTube, but this needs HTML, CSS anf JAVA knowledge.
- Otherwise we can craete an UI in streamlit (easy for me) where I can share the video id and it will fetch the transcript of that YouTube video and can answer any query regarding the video by seeing the transcript. 

Example:
- Want to know if in any podcast or yt video something about AI related topic?
- This RAG based system can answer that.

### Load API Key for OpenAI Models

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### Load Libraries
- Required libraries are already installed in the virtual environment via `requirements.txt`.

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\koyel\AppData\Local\Temp\ipykernel_8276\3829014579.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### Step 1a - Indexing (Document Ingestion)

- For loadiing transcripts in certain language, transcript should be present in that language, otherwise code will break.
- Here as `video_id` paste only the ID from video url, not full URL.
- In the below code, I pasted id of https://www.youtube.com/watch?v=BUTjcAjfMgY&t=167s. 
- For this parameter `language` should be `hi` if video is in hindi, otherwise code will break, as this video may not have transcript in english or other language. 
- By default this `language` parameter is `en` (english), so if any video is in english, then no need to write it. Actually it returns the `best one`. 

In [12]:
video_id = "BUTjcAjfMgY"

try:
    # if you don't care which language, this returns the "best" one
    yt_api = YouTubeTranscriptApi()
    transcript_list = yt_api.fetch(
        video_id = video_id,
        languages = ("en",)
        )

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)
    
except TranscriptsDisabled:
    print("No captions available for this video.")

Hey everyone, I'm Sha. In this video, I'll be covering machine learning fundamentals that every AI engineer needs to know. While you'll find endless textbooks and articles on this subject, my goal with this video is to give builders a short and accessible guide to the most critical concepts in machine learning. Here, I'm going to focus on five key points. I'll first talk about intelligence and world models. Then I'll talk about three different ways computers can learn world models through machine learning, deep learning, and reinforcement learning. Finally, I'll talk about the most important ingredient in machine learning, which is data. So let's talk about intelligence. Intelligence requires understanding how the world works. But of course, the world is a big and complicated place. So in order to understand it, you need to develop models of the world so that you can compress the complicated and vast reality that we live in into something you can fit into your head. And put simply, a m

Below we can see transcript based on timestamp.

In [13]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="Hey everyone, I'm Sha. In this video,", start=0.0, duration=4.319), FetchedTranscriptSnippet(text="I'll be covering machine learning", start=2.48, duration=4.56), FetchedTranscriptSnippet(text='fundamentals that every AI engineer', start=4.319, duration=5.041), FetchedTranscriptSnippet(text="needs to know. While you'll find endless", start=7.04, duration=4.96), FetchedTranscriptSnippet(text='textbooks and articles on this subject,', start=9.36, duration=4.56), FetchedTranscriptSnippet(text='my goal with this video is to give', start=12.0, duration=5.199), FetchedTranscriptSnippet(text='builders a short and accessible guide to', start=13.92, duration=5.76), FetchedTranscriptSnippet(text='the most critical concepts in machine', start=17.199, duration=4.401), FetchedTranscriptSnippet(text="learning. Here, I'm going to focus on", start=19.68, duration=4.16), FetchedTranscriptSnippet(text="five key points. I'll first talk about", st

In [14]:
print(len(transcript_list))

864


Transcript got loaded on timestamp basis. We merged all pieces of transcript into one.

In [15]:
transcript

"Hey everyone, I'm Sha. In this video, I'll be covering machine learning fundamentals that every AI engineer needs to know. While you'll find endless textbooks and articles on this subject, my goal with this video is to give builders a short and accessible guide to the most critical concepts in machine learning. Here, I'm going to focus on five key points. I'll first talk about intelligence and world models. Then I'll talk about three different ways computers can learn world models through machine learning, deep learning, and reinforcement learning. Finally, I'll talk about the most important ingredient in machine learning, which is data. So let's talk about intelligence. Intelligence requires understanding how the world works. But of course, the world is a big and complicated place. So in order to understand it, you need to develop models of the world so that you can compress the complicated and vast reality that we live in into something you can fit into your head. And put simply, a 

### Step 1b - Indexing (Text Splitting)
- As video is very long, so we have to split it.

In [16]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, 
    chunk_overlap = 200
    )

chunks = splitter.create_documents([transcript])

In [17]:
len(chunks)

41

In [18]:
chunks[0]

Document(metadata={}, page_content="Hey everyone, I'm Sha. In this video, I'll be covering machine learning fundamentals that every AI engineer needs to know. While you'll find endless textbooks and articles on this subject, my goal with this video is to give builders a short and accessible guide to the most critical concepts in machine learning. Here, I'm going to focus on five key points. I'll first talk about intelligence and world models. Then I'll talk about three different ways computers can learn world models through machine learning, deep learning, and reinforcement learning. Finally, I'll talk about the most important ingredient in machine learning, which is data. So let's talk about intelligence. Intelligence requires understanding how the world works. But of course, the world is a big and complicated place. So in order to understand it, you need to develop models of the world so that you can compress the complicated and vast reality that we live in into something you can fit

In [23]:
# first chunk (python index 0-40 for 41 chunks)
chunks[0]

Document(metadata={}, page_content="Hey everyone, I'm Sha. In this video, I'll be covering machine learning fundamentals that every AI engineer needs to know. While you'll find endless textbooks and articles on this subject, my goal with this video is to give builders a short and accessible guide to the most critical concepts in machine learning. Here, I'm going to focus on five key points. I'll first talk about intelligence and world models. Then I'll talk about three different ways computers can learn world models through machine learning, deep learning, and reinforcement learning. Finally, I'll talk about the most important ingredient in machine learning, which is data. So let's talk about intelligence. Intelligence requires understanding how the world works. But of course, the world is a big and complicated place. So in order to understand it, you need to develop models of the world so that you can compress the complicated and vast reality that we live in into something you can fit

In [ ]:
# last chunk (python index 0-40 for 41 chunks)
chunks[40]

Document(metadata={}, page_content="learning is deep learning, which involves using neural networks to learn useful features and mappings from raw data. And a lot of times deep learning is combined with reinforcement learning which allows computers to learn by interacting with the world through trial and error. And finally, although there's a lot of math and fancy algorithms and machine learning, training good models comes down to having good data. And what this means is you need high volume, highquality training data sets. Although there were many things I couldn't cover in this short overview here, I hope it gave you more clarity around some buzzwords in machine learning and the confidence to continue your learning on your own. Toward that end, if you have any questions or suggestions for future topics you want me to cover, please let me know in the comments below. And as always, thank you so much for your time and thanks for watching.")

### Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Space)

In [24]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [25]:
# ids against each chunk
vector_store.index_to_docstore_id 

{0: 'c546b7ee-a46a-4b4f-91d1-bb4a90967947',
 1: 'b0cce857-b2ac-450f-a873-721cf65bee79',
 2: 'd11d7f56-d593-433d-8ebb-b49f133bd5c3',
 3: '0fa207a9-4c03-455a-96b3-afdfdee11e12',
 4: '79166306-4035-49b5-8846-8abe26944919',
 5: '1bf0616b-dce8-4892-ac30-055ebe0a85ec',
 6: 'c41de8cf-8da1-423b-a7e8-c200502e8a57',
 7: '9ad09952-2633-48e0-a4f1-2ba05ecc4e14',
 8: '0c851088-d8e9-4bf6-bc10-cca73c855d51',
 9: 'b168abaa-b904-4a95-82f5-6e03ea70d30e',
 10: 'b96439ed-3406-4d81-bd1d-5258d46ae9d5',
 11: '35140be0-307c-4e59-80c9-9271b6535b73',
 12: '4509020e-a3d9-401a-b157-8ddf7b72362e',
 13: '9da2a51a-484d-43e1-b2ce-7c3731786811',
 14: '5bf2e7bc-d6b1-47ec-80bb-7b7083d5941c',
 15: 'c9dc51ea-9d37-4c02-bfe8-2b1f322e1f1f',
 16: 'c407683b-afb9-4dce-84de-4b7dca6eca57',
 17: '5d87345d-41d7-4d12-8bea-200c650fe77b',
 18: '1f9b0e7a-bd7e-486f-b451-b5dabae6d82c',
 19: '41c8e194-e843-4b92-9e89-1959b972e111',
 20: '5ba455b3-0318-43c9-bf20-a73ab79015f9',
 21: 'ed8c3045-ec2c-45b7-862c-8cad5ed313a1',
 22: '7e0dc9bc-2732-

In [26]:
# put one id here to see a particular chunk
vector_store.get_by_ids(['c5e3d0e0-f9c5-478c-8939-ac595551e6a7']) # chunk id 36 (37th chunk)

[Document(id='c5e3d0e0-f9c5-478c-8939-ac595551e6a7', metadata={}, page_content="batch different examples into a group when computing this objective and updating the model parameters. For most of this video, I've talked about algorithms for optimizing parameter values and loss functions and neural networks and sophisticated fancy math. And this might give you the impression that algorithms are the most important thing when it comes to machine learning. However, that is very much not the case because at the end of the day, we are simply fitting models to our data. So if you have bad data, it doesn't matter how sophisticated your algorithm is or how much you tune your hyperparameters. If you're fitting a model to bad data, you're going to have a bad model. That's why I wanted to finish this talk to review what makes good data. And so there really two properties that make data good. The first is quantity. All things equal, more data are better than less data. When you don't have enough dat

### Step 2 - Retrieval
- This is just retrieval part. 
- Here we will get return relevant documents.

In [27]:
# return 4 most similar chunks
retriever = vector_store.as_retriever(
    search_type = "similarity", 
    search_kwargs = {"k": 4}
    ) 

In [28]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022E30BA2A50>, search_kwargs={'k': 4})

- Ask question related to uploaded video.
- In retriever input is question, output is list of documents from above 37 chunks.

In [29]:
retriever.invoke('What is deepmind') 

[Document(id='d14c54e1-03a4-41f8-8c0d-b9dd4f8574c4', metadata={}, page_content="And the central reason for this is that with reinforcement learning, models are not bound by human labeling or expertise. In supervised learning, the performance of our model is going to be bound by our ability to label highquality data or our existing knowledge or expertise. However, with reinforcement learning, since the model is interacting with the environment on its own, it can find its own ways of solving problems and potentially surpass even expert human performance. A real world example of this is Alph Go, which is a deep learning model that Deep Mind developed to play the game of Go. So there's actually a whole movie about this, but in this nature paper, they showed this plot here comparing a reinforcement learningbased deep neural network, a supervised learning based deep neural network, and the ELO rating of this Go Grandmaster. And we can see that the Grandmaster has just below a 4,000 ELO ratin

### Step 3 - Augmentation
- In this part retrieved related documents will be augmned with prompt.

In [30]:
llm = ChatOpenAI(
    model='gpt-4o-mini', 
    temperature = 0.2
    )

- This is the prompt template with placeholder.

In [31]:
prompt = PromptTemplate(
    template = """
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.
        
        {context}
        Question:  {question}
        """,
    input_variables = ['context', 'question']
)

- Here is the question and retrieved documents.
- Here is no augmentation yet.

In [32]:
question = "is the topic of aliens discussed in this video? if yes, then what was discussed?"
retrieved_docs = retriever.invoke(question)

In [33]:
retrieved_docs

[Document(id='c546b7ee-a46a-4b4f-91d1-bb4a90967947', metadata={}, page_content="Hey everyone, I'm Sha. In this video, I'll be covering machine learning fundamentals that every AI engineer needs to know. While you'll find endless textbooks and articles on this subject, my goal with this video is to give builders a short and accessible guide to the most critical concepts in machine learning. Here, I'm going to focus on five key points. I'll first talk about intelligence and world models. Then I'll talk about three different ways computers can learn world models through machine learning, deep learning, and reinforcement learning. Finally, I'll talk about the most important ingredient in machine learning, which is data. So let's talk about intelligence. Intelligence requires understanding how the world works. But of course, the world is a big and complicated place. So in order to understand it, you need to develop models of the world so that you can compress the complicated and vast realit

- Here all page contents will be merged to get cosolidated document.

In [34]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

- Here the final prompt is getting generated.

In [37]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n\n        Hey everyone, I'm Sha. In this video, I'll be covering machine learning fundamentals that every AI engineer needs to know. While you'll find endless textbooks and articles on this subject, my goal with this video is to give builders a short and accessible guide to the most critical concepts in machine learning. Here, I'm going to focus on five key points. I'll first talk about intelligence and world models. Then I'll talk about three different ways computers can learn world models through machine learning, deep learning, and reinforcement learning. Finally, I'll talk about the most important ingredient in machine learning, which is data. So let's talk about intelligence. Intelligence requires understanding how the world works. But of course, the world is a big and complicated place. So in o

### Step 4 - Generation
- Here actually answer will be generated with LLM and generated final prompt.

In [38]:
answer = llm.invoke(final_prompt)
print(answer)

content="I don't know." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 778, 'total_tokens': 782, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bbc23ca3d7', 'id': 'chatcmpl-DzRX7xogfZvxfQIMk5lMQtgQZUilo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f430a-ea09-7993-a8ea-7ba510af3f50-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 778, 'output_tokens': 4, 'total_tokens': 782, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [39]:
print(answer.content)

I don't know.


### Building a Chain
- Building chain as invoking all components - indexing, retrival, generatipom, augmenattion separately is not good practice. It's not standardized way.
- It should be using `invoke` method, so that method will be called just once and all steps will be done, so making this chain here.

In [40]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [41]:
# put above cosolidaing formula in a function format.
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [42]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [43]:
parallel_chain.invoke('who is Demis')

{'context': "reality. So there are two key aspects of this. The first is accuracy, which is the correctness of your data. For example, if your data set says someone is 35 years old when in reality the person is actually 53 years old, that is bad data. Another example is if your data says someone makes $12,000 a year when in fact they make $12,000 a month. That's another example of bad inaccurate data. The other aspect is diversity, which is basically the representativeness of your data. For example, if you have some kind of SAS product and you're trying to predict customer churn and you take data from a bunch of prousers of your SAS product and you use it to develop a customer churn model, this is completely fine if you're trying to predict churn of users of your pro plan. However, if you have an enterprise customer, this model probably won't be helpful in predicting the churn of an enterprise. In other words, it's not only important that your data are correct and accurately represent 

In [44]:
parser = StrOutputParser()

In [45]:
main_chain = parallel_chain | prompt | llm | parser

In [46]:
main_chain.invoke('Can you summarize the video')

'In the video, Sha discusses machine learning fundamentals essential for AI engineers, focusing on five key points. She begins by explaining the concept of intelligence and the need for world models to understand the complexities of reality. She then outlines three methods for learning these models: machine learning, deep learning, and reinforcement learning. Sha emphasizes the importance of data in training effective models, stating that high volume and high-quality datasets are crucial. She also touches on various algorithms for optimizing model parameters and highlights that good data is more important than sophisticated algorithms. Finally, she encourages viewers to continue their learning and invites questions or topic suggestions.'

### TODO:
- Repeat the exercise on prompt generation with new question: "Is the topic of nuclear fusion discussed in this video? if yes, then what was discussed?"
- Can try with different video as well, with different question obviously.

# Improvements

1. UI based enhancements 
(Final product can be seen as awebsite, for this YTChatbot, we can create chrome plugin use has to install and when he will open YT he can see it and chat using it)

2. Evaluation
   a. Ragas
   b. LangSmith

3. Indexing
   a. Document Ingestion 
   (YT transcript is auto-generated, we can keep a step to fix all bugs in auto-generated transcript; also all videos are not in english or our desired language, we can keep a step to translate transcript from one language to another)

   b. Text Splitting 
   (Instead of character splitter can use semantic chunker)

   c. Vector Store 
   (instead of faiss, can try pinecone)

4. Retrieval
   a. Pre-retrieval
      i. Query rewriting using LLM
      ii. Multi-query generation
      iii. Domain aware routing
   b. Duing Retrieval
      i. MMR
      ii. Hybrid retrieval 
      (Semantic Search + Key-word Search)

      iii. Reranking 
      (creation of ranks of all retrived documents; reranked using LLM)

   c. Post-retrieval
      i. Contextual Compression
      (Can keep only useful part of content, can remove other parts)

5. Augmentation
   a. Prompt Templating 
   b. Answer grounding
   (Don't create answer from yourself, don't halluciante, answer based on fact only)

   c. Context window optimization
   (Trim generated context in such a way that only useful info will stay, otherwise if token limits will cross then LLM will not answer - Is token limit different for input and output?)

6. Generation
   a. Answer with Citation
   b. Guard railing
   (Prevent LLM from generating improper/LLM answers)

7. System Design
   a. Multimodal
   (Can work with text, audio, video etc)

   b. Agentic
   (In process of answering question if needed it can do internet browing apart from understanding context and answer generation)

   c. Memory based
   (Can keep old memory in memory)

These are part of Advanced RAG.